# 16 Cause and effect, epidemiology and bioequivalence: lalonde, JOBS II, Willeberg, Selvin, Schütz et al.

- `table.compare_balanced_groups_of` on **lalonde** (LaLonde 1986; Dehejia & Wahba 1999; 185 men in the
  NSW job training and 429 from a survey): inverse probability weighting (the effect for everyone) against
  R WeightIt (`weightit`, `glm_weightit`) and a statsmodels logistic propensity score with the weighted means
  written out; the balance table against R cobalt `bal.tab`; 1:1 nearest-neighbour matching (with and without
  a caliper) against R MatchIt with `lm` and pair-clustered standard errors (sandwich `vcovCL`).
- `table.explain(through=)`, causal mediation on **JOBS II** (Vinokur, Price & Schul 1995; the `jobs` data of
  R mediation): against statsmodels `Mediation` with the same seed (the parent), against the exact limit of
  the simulation for two linear models (the product of coefficients), and against R `mediation::mediate`
  (a second, independent simulation). simplydata's own two linear models (`explain`) give the product of
  coefficients too, compared with base R `lm`.
- `table.compare_risks(show="attributable fractions")` on **dry cat food and urinary disease** (Willeberg
  1977, the first example of R epiR `epi.2by2`): against epiR, against the printed values in epiR's help, and
  against the textbook formulas (Rothman, Greenland & Lash 2008) in NumPy.
- `table.rate_of(standard_population=)` and `table.compare_rates(standard_rates=)` on **Down syndrome by birth
  order and mother's age** in Michigan (Selvin 2004; the example of R epitools): direct standardisation with
  Fay & Feuer's (1997) gamma interval and indirect standardisation (observed / expected, SMR) with its exact
  Poisson interval, against R epitools (`ageadjust.direct`, `ageadjust.indirect`; a different package from
  epiR, which simplydata runs), R `poisson.test` and the formulas in SciPy.
- `table.compare_averages_of(period=, same_within=)`, the 2 x 2 crossover, on the three **bioequivalence
  reference datasets** of Schütz, Labes & Fuglsang (2014): against their published ratios and 90 %
  intervals, statsmodels OLS and R `lm` with the EMA model (sequence, subject within sequence, period and
  formulation, all fixed, on log AUC).

**Only simplydata runs in this notebook.** The reference values (`references/16_cause_and_effect.csv`, from
`scripts/references/16_cause_and_effect.py`); every check prints its source.

References and tolerances:
- **Printed values**: half a unit of the last printed digit.
- **Closed formulas** (NumPy/SciPy written out, R `poisson.test`, epitools): a relative difference of 1e-10.
- **Least squares** (the crossover against statsmodels OLS and R `lm`; simplydata's product of coefficients
  against R `lm`): 1e-9.
- **Models fitted by iteration** (propensity scores by logistic regression in R WeightIt, MatchIt, cobalt,
  statsmodels; epiR): 1e-6.
- **Simulations**: statsmodels `Mediation` with the same seed must give the same numbers (1e-10). Against
  the exact limit and against R's own simulation, the allowed difference is 4 Monte Carlo standard errors of
  the simulated average (sd of the 1000 simulated values / sqrt(1000), times sqrt(2) when both sides are
  simulations), worked out in the reference script from statsmodels' simulation.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "16_cause_and_effect"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## lalonde: `compare_balanced_groups_of`

The propensity score is a logistic regression of `treat` on age, schooling, race, marriage, no degree and
earnings in 1974 and 1975. With `method="weighting"` (the default) each man gets the weight 1/score (trained)
or 1/(1 - score) (not trained), the effect for everyone (ATE). One full printed answer first.

In [2]:
lalonde = quiet(sdata.table, "../data/fetched/lalonde.csv", separator=",")
COLUMNS = ["age", "educ", "race", "married", "nodegree", "re74", "re75"]
lalonde.compare_balanced_groups_of("re78", for_each_group_in="treat", allowing_for=COLUMNS)

Note: "1" compared against "0" (the first in sorted order, as R); choose another with against=.


Groups made alike in age, educ, race, married, nodegree, re74 and re75 (weighting), then compared: "re78" in "1" against "0" (614 rows: 185 and 429).
Propensity score: each row's chance of being in "1" given age, educ, race, married, nodegree, re74 and re75, from logistic regression (R glm(family = binomial)).
Overlap: the scores run from 0.0250 to 0.853 in "1" and from 0.00908 to 0.789 in "0"; 8 rows of "1" and 57 of "0" lie outside the other group's range (no row like them in the other group: their effect is guessed from the model).
Balance (standardised differences, positive: higher in "1"; within 0.1 is the usual guide, Austin 2009):
  column        before  after
  age           -0.242  -0.168
  educ           0.045   0.130
  race: black    1.671   0.130
  race: hispan  -0.277   0.016
  race: white   -1.408  -0.138
  married       -0.721  -0.210
  nodegree       0.235  -0.116
  re74          -0.596  -0.274
  re75          -0.287  -0.158
Note: age, educ, race: black, race: white, ma

,term,estimate,standard error,estimate 95 % CI low,estimate 95 % CI high,p-value,effect for,rows,method
1,1 against 0,224.676308,876.193186,-1492.630779,1941.983396,0.797624,everyone (ATE),614,weighting on the propensity score (R WeightIt)


In [3]:
DS = "lalonde"
found = read_row(quiet(lalonde.compare_balanced_groups_of, "re78", for_each_group_in="treat", allowing_for=COLUMNS))
checks.check_values(DS, "compare_balanced_groups_of (weighting, everyone)",
                    {"effect": found["estimate"], "standard error": found["standard error"], "p-value": found["p-value"]})

agrees  lalonde | compare_balanced_groups_of (weighting, everyone) | effect: simplydata 224.676308271, statsmodels GLM + weighted means 224.676309104; relative difference 3.71e-09 (allowed 1e-06)
      Source: statsmodels GLM + weighted means (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=GLM
agrees  lalonde | compare_balanced_groups_of (weighting, everyone) | effect: simplydata 224.676308271, R WeightIt lm_weightit 224.676308271; relative difference 0.00e+00 (allowed 1e-06)
      Source: R WeightIt lm_weightit (run reference) - https://search.r-project.org/CRAN/refmans/WeightIt/html/lm_weightit.html (version 2.1.0)
agrees  lalonde | compare_balanced_groups_of (weighting, everyone) | standard error: simplydata 876.193185589, R WeightIt lm_weightit 876.193185589; relative difference 0.00e+00 (allowed 1e-06)
      Source: R WeightIt lm_weightit (run reference) - https://search.r-project.org/CRAN/refmans/WeightIt/html/lm_weightit.html (version 2.1.0)
agrees  lalonde |

4

### The balance table: standardised differences before and after weighting (cobalt)

In [4]:
balance = rows(lalonde.compare_balanced_groups_of, "re78", for_each_group_in="treat", allowing_for=COLUMNS, show="balance")
for row in balance:
    checks.check_values(DS, "compare_balanced_groups_of (show=balance)",
                        {f"{row['column']} difference before": row["difference before"],
                         f"{row['column']} difference after": row["difference after"]})

agrees  lalonde | compare_balanced_groups_of (show=balance) | age difference before: simplydata -0.241903622926, R cobalt bal.tab -0.241903622926; relative difference 1.15e-16 (allowed 1e-06)
      Source: R cobalt bal.tab (run reference) - https://search.r-project.org/CRAN/refmans/cobalt/html/bal.tab.html (version 5.0.0)
agrees  lalonde | compare_balanced_groups_of (show=balance) | age difference after: simplydata -0.167567586402, R cobalt bal.tab -0.167567586402; relative difference 4.97e-16 (allowed 1e-06)
      Source: R cobalt bal.tab (run reference) - https://search.r-project.org/CRAN/refmans/cobalt/html/bal.tab.html (version 5.0.0)
agrees  lalonde | compare_balanced_groups_of (show=balance) | educ difference before: simplydata 0.0447550851109, R cobalt bal.tab 0.0447550851109; relative difference 1.40e-15 (allowed 1e-06)
      Source: R cobalt bal.tab (run reference) - https://search.r-project.org/CRAN/refmans/cobalt/html/bal.tab.html (version 5.0.0)
agrees  lalonde | compare_ba

### Matching: 1:1 nearest neighbour, the effect among the trained men (ATT)

R MatchIt `matchit(method = "nearest", distance = "glm")`, then `lm(y ~ t, weights = weights)` on the matched
men with standard errors clustered on the pairs (sandwich `vcovCL(cluster = ~subclass)`), the analysis the
MatchIt documentation recommends; then with a caliper of 0.2 standard deviations of the logit of the score.

In [5]:
for caliper in (None, 0.2):
    M = "compare_balanced_groups_of (matching, trained)" + ("" if caliper is None else ", match_within=0.2")
    extra = {} if caliper is None else {"match_within": caliper}
    found = read_row(quiet(lalonde.compare_balanced_groups_of, "re78", for_each_group_in="treat", allowing_for=COLUMNS,
                           method="matching", effect_for=1, **extra))
    checks.check_values(DS, M, {"effect": found["estimate"], "standard error": found["standard error"]})

agrees  lalonde | compare_balanced_groups_of (matching, trained) | effect: simplydata 894.36748027, R MatchIt + lm + vcovCL 894.36748027; relative difference 0.00e+00 (allowed 1e-06)
      Source: R MatchIt + lm + vcovCL (run reference) - https://cran.r-project.org/package=MatchIt (version 4.8.2)
agrees  lalonde | compare_balanced_groups_of (matching, trained) | standard error: simplydata 705.229764062, R MatchIt + lm + vcovCL 705.229764062; relative difference 0.00e+00 (allowed 1e-06)
      Source: R MatchIt + lm + vcovCL (run reference) - https://cran.r-project.org/package=MatchIt (version 4.8.2)


agrees  lalonde | compare_balanced_groups_of (matching, trained), match_within=0.2 | effect: simplydata 1770.60290641, R MatchIt + lm + vcovCL 1770.60290641; relative difference 0.00e+00 (allowed 1e-06)
      Source: R MatchIt + lm + vcovCL (run reference) - https://cran.r-project.org/package=MatchIt (version 4.8.2)
agrees  lalonde | compare_balanced_groups_of (matching, trained), match_within=0.2 | standard error: simplydata 924.234347046, R MatchIt + lm + vcovCL 924.234347046; relative difference 0.00e+00 (allowed 1e-06)
      Source: R MatchIt + lm + vcovCL (run reference) - https://cran.r-project.org/package=MatchIt (version 4.8.2)


## JOBS II: `explain(through=)`, causal mediation

Does the job-search workshop (`treat`) lower depression six months later (`depress2`) through job-search
confidence (`job_seek`), allowing for economic hardship, sex and age? Two linear models, 1000 quasi-Bayesian
simulations (Imai, Keele & Tingley 2010), `with_seed=1`. No published estimate exists for exactly this model
(Tingley et al. 2014 analyse JOBS II with noncompliance), so the references are the parent with the same seed,
the exact limit of the simulation and R's own simulation.

In [6]:
jobs = quiet(sdata.table, "../data/fetched/jobs_ii.csv", separator=",")
OTHERS = ["econ_hard", "sex", "age"]
jobs.explain("depress2", using="treat", through="job_seek", allowing_for=OTHERS, with_seed=1)

How much of the effect of "treat" (1 against 0) on "depress2" goes through "job_seek", allowing for econ_hard, sex and age (899 rows):
  indirect effect (through job_seek): -0.0163 (95 % interval -0.0458 to 0.0114)
  direct effect: -0.0392 (95 % interval -0.128 to 0.0380)
  total effect: -0.0555 (95 % interval -0.154 to 0.0275)
  share through job_seek: 0.250 (95 % interval -1.94 to 3.28); the total effect is not clear here, so this share says little.
If nothing went through "job_seek", an indirect effect this far from 0 would turn up in about 29 of 100 studies like this one (p = 0.286). This does not show that nothing goes through "job_seek".
These are effects of "treat" only if nothing that was not measured affects both "treat" and "depress2", nor both "job_seek" and "depress2" (allowing_for= takes the measured ones into account), "treat" came before "job_seek", and "job_seek" came before "depress2". A randomised "treat" makes the first true for "treat", never for "job_seek".
Method:

,term,estimate,95 % CI low,95 % CI high,p-value
1,indirect effect (through job_seek),-0.016316,-0.045754,0.011438,0.286
2,direct effect,-0.039181,-0.128117,0.038033,0.384
3,total effect,-0.055496,-0.153547,0.027524,0.212
4,share through job_seek,0.249532,-1.943755,3.280017,0.370


In [7]:
DS, M = "JOBS II", "explain (through=)"
found = {row["term"]: row for row in rows(jobs.explain, "depress2", using="treat", through="job_seek",
                                          allowing_for=OTHERS, with_seed=1)}
TERMS = {"indirect effect": "indirect effect (through job_seek)", "direct effect": "direct effect",
         "total effect": "total effect"}
checks.check_values(DS, M, {f"{name} {column}": found[term][column] for name, term in TERMS.items()
                            for column in ("estimate", "95 % CI low", "95 % CI high")})
# the product of coefficients from simplydata's own two linear models
a = {row["term"]: row for row in rows(jobs.explain, "job_seek", using=["treat"] + OTHERS)}["treat"]["effect"]
b = {row["term"]: row for row in rows(jobs.explain, "depress2", using=["treat", "job_seek"] + OTHERS)}["job_seek"]["effect"]
checks.check(DS, "R lm (check of the limit)", "a * b", a * b)

agrees  JOBS II | explain (through=) | indirect effect estimate: simplydata -0.0163155082561, statsmodels Mediation (same seed) -0.0163155082561; relative difference 2.34e-15 (allowed 1e-10)
      Source: statsmodels Mediation (same seed) (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=Mediation
agrees  JOBS II | explain (through=) | indirect effect estimate: simplydata -0.0163155082561, exact limit (product of coefficients, statsmodels OLS) -0.0157446450289; difference 5.71e-04 (allowed 2e-03)
      Source: exact limit (product of coefficients, statsmodels OLS) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/16_cause_and_effect.py
agrees  JOBS II | explain (through=) | indirect effect estimate: simplydata -0.0163155082561, R mediation::mediate -0.0158506524061; difference 4.65e-04 (allowed 3e-03)
      Source: R mediation::mediate (run reference) - https://search.r-project.org/CRAN/refmans/mediation/html/mediate.html

1

## Attributable fractions: dry cat food and feline urinary disease (Willeberg 1977)

13 of 2176 cats on dry food and 5 of 3354 on other food had the disease. epiR's help prints the attributable
fraction among the exposed 75.05 % (95 % CI 32.91 to 90.72) and in the population 54.20 %. simplydata's
printout says "Wald intervals from the risk ratio", so the formulas are written out that way (Rothman,
Greenland & Lash 2008): AFe = (RR - 1) / RR with the Wald interval of RR, exp(log RR +/- 1.96 SE),
carried through; PAF = (R - R0) / R, with R the risk in all cats and R0 in the unexposed, and its Wald
interval on log(1 - PAF) (Jewell 2004, *Statistics for Epidemiology*, section 7.1; the formula epiR itself
uses for stratified tables). epiR, run with one table, reports under the name `AFRisk.strata.wald` the
interval from the **score** interval of the risk ratio (1.49 to 10.78, its `RR.strata.score`; the Wald
interval is 1.43 to 11.23), and for the population fraction the interval of its crude method (the extreme
combinations of the limits of the two risks), which simplydata passes on (FINDINGS A5).

In [8]:
cats = load("cats")
af = cats.compare_risks("flutd", for_each_group_in="food", against="other food", show="attributable fractions")
af

"flutd" = "yes": 13 of 2176 in dry cat food and 5 of 3354 in other food. Exposed: dry cat food; not exposed: other food (against=).


Among those in dry cat food, 75.0 % of the events go with the exposure (95 % CI 32.9 % to 90.7 %): the attributable fraction among the exposed, (risk ratio - 1) / risk ratio, with risk ratio 4.01.
In the whole population of these rows, 54.2 % of the events go with the exposure (95 % CI 32.4 % to 74.9 %): the population attributable fraction, which also depends on how many are exposed (here 2176 of 5530).
Crude: not adjusted for other columns (a confounder can make it too big or too small).
These fractions are about events that would not have happened without the exposure only if "food" causes them; a link alone does not show that.
Method: R epiR::epi.2by2(method = "cohort.count"); intervals as epiR gives them for one table: among the exposed from the score interval of the risk ratio, in the population from the exact intervals of the two risks (Rothman, Greenland & Lash 2008). The risks must come from a cohort or a cross-sectional study: in a case-control study the risks cannot be estim

,measure,group,against,estimate,estimate 95 % CI low,estimate 95 % CI high,method
1,attributable fraction among the exposed,dry cat food,other food,0.750470,0.329129,0.907214,"R epiR::epi.2by2(method = ""cohort.count""); int..."
2,attributable fraction in the population,dry cat food,other food,0.542006,0.323759,0.749138,"R epiR::epi.2by2(method = ""cohort.count""); int..."


In [9]:
DS, M = "dry cat food and FLUTD", "compare_risks (attributable fractions)"
exposed, population = read_rows(af)[:2]
ours = {"AF among the exposed": exposed["estimate"], "AF 95 % CI low": exposed["estimate 95 % CI low"],
        "AF 95 % CI high": exposed["estimate 95 % CI high"], "PAF": population["estimate"],
        "PAF 95 % CI low": population["estimate 95 % CI low"], "PAF 95 % CI high": population["estimate 95 % CI high"]}
scale = 100 if ours["AF among the exposed"] > 1 else 1  # simplydata may give percents
checks.check_values(DS, M, {f"{quantity} (percent)": value * 100 / scale for quantity, value in ours.items()})
checks.check_values(DS, M, {quantity: value / scale for quantity, value in ours.items()})

agrees  dry cat food and FLUTD | compare_risks (attributable fractions) | AF among the exposed (percent): simplydata 75.0470161919, epiR help, printed 75.05; difference 2.98e-03 (allowed 5e-03)
      Source: epiR help, printed (printed) - https://search.r-project.org/CRAN/refmans/epiR/html/epi.2by2.html
agrees  dry cat food and FLUTD | compare_risks (attributable fractions) | AF 95 % CI low (percent): simplydata 32.9128862567, epiR help, printed 32.91; difference 2.89e-03 (allowed 5e-03)
      Source: epiR help, printed (printed) - https://search.r-project.org/CRAN/refmans/epiR/html/epi.2by2.html
agrees  dry cat food and FLUTD | compare_risks (attributable fractions) | AF 95 % CI high (percent): simplydata 90.721354725, epiR help, printed 90.72; difference 1.35e-03 (allowed 5e-03)
      Source: epiR help, printed (printed) - https://search.r-project.org/CRAN/refmans/epiR/html/epi.2by2.html
agrees  dry cat food and FLUTD | compare_risks (attributable fractions) | PAF (percent): simplyda

12

## Standardised rates: Down syndrome by birth order, allowing for the mother's age (Selvin 2004)

Cases and live births in Michigan by birth order (1 to 5+) and mother's age (6 bands). Direct
standardisation to the whole table's births by age, per 100,000 births; the epitools help prints 92.3 (80.4 to
105.8) for the first born (Fay & Feuer's gamma interval). Indirect standardisation: observed / expected
against the pooled rates by age, with the exact Poisson interval and p-value (R `poisson.test`; SciPy's
chi-squared form of Garwood's interval).

In [10]:
down = quiet(sdata.table, "../data/down_syndrome.csv", separator=",")
direct = down.rate_of("cases", time="births", for_each_group_in="birth_order", allowing_for="mother_age",
                      standard_population="whole table", per=100000)
direct

"cases" per 100000 "births", standardised for "mother_age" (6 bands) to the whole table's "births" in each mother_age:
  1: adjusted 92.3 (95 % CI 80.4 to 106); crude 56.3.
  2: adjusted 91.2 (95 % CI 82.4 to 101); crude 67.6.
  3: adjusted 85.1 (95 % CI 77.2 to 94.2); crude 83.3.
  4: adjusted 92.7 (95 % CI 80.0 to 115); crude 115.
  5+: adjusted 75.5 (95 % CI 67.7 to 188); crude 167.
  Unknown: adjusted 7.96 (95 % CI 6.96 to 9.08); crude 7.96.
The adjusted rate is the rate each group would have with the make-up of the standard population in "mother_age", so groups with different make-ups can be compared. Its size depends on the standard chosen: compare adjusted rates only with others made with the same standard.
Method: direct standardisation, R epiR::epi.directadj() (the same numbers as R epitools::ageadjust.direct()): the adjusted rate with Fay & Feuer's (1997) gamma interval; the crude rate with its exact Poisson interval.


,group,events,time,crude rate,crude rate 95 % CI low,crude rate 95 % CI high,adjusted rate,adjusted rate 95 % CI low,adjusted rate 95 % CI high,per
1,1,412.0,731177.0,56.347505,51.037236,62.060172,92.304526,80.441750,105.763346,100000 births
2,2,490.0,724639.0,67.619877,61.764427,73.880809,91.174117,82.361212,100.867576,100000 births
3,3,474.0,568819.0,83.330550,75.996665,91.181122,85.069221,77.183453,94.223570,100000 births
4,4,413.0,357727.0,115.451168,104.583739,127.141062,92.717865,80.009993,114.669445,100000 births
5,5+,740.0,442811.0,167.114186,155.288960,179.601126,75.529007,67.702120,188.300173,100000 births
6,Unknown,225.0,2825173.0,7.964114,6.957409,9.075555,7.964114,6.957409,9.075555,100000 births


In [11]:
DS, M = "Down syndrome by birth order", "rate_of (standard_population=)"
for row in read_rows(direct):
    checks.check_values(DS, f"{M}, birth order {row['group']}",
                        {quantity: row[quantity] for quantity in ("crude rate", "adjusted rate", "adjusted rate 95 % CI low",
                                                                  "adjusted rate 95 % CI high")})

agrees  Down syndrome by birth order | rate_of (standard_population=), birth order 1 | crude rate: simplydata 56.3475054604, SciPy, Fay & Feuer formula 56.3475054604; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy, Fay & Feuer formula (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  Down syndrome by birth order | rate_of (standard_population=), birth order 1 | crude rate: simplydata 56.3475054604, R epitools ageadjust.direct 56.3475054604; relative difference 0.00e+00 (allowed 1e-10)
      Source: R epitools ageadjust.direct (run reference) - https://search.r-project.org/CRAN/refmans/epitools/html/ageadjust.direct.html (version 0.5.10.1)
agrees  Down syndrome by birth order | rate_of (standard_population=), birth order 1 | adjusted rate: simplydata 92.3045255332, SciPy, Fay & Feuer formula 92.3045255332; relative difference 1.54e-16 (allowed 1e-10)
      Source: SciPy, Fay & Feuer formula (run reference) - https://docs.scipy.org/d

In [12]:
M = "compare_rates (standard_rates=)"
cases = {row["mother_age"]: row["total of cases"] for row in rows(down.find_total_of, "cases", for_each_group_in="mother_age")}
births = {row["mother_age"]: row["total of births"] for row in rows(down.find_total_of, "births", for_each_group_in="mother_age")}
rates = {age: cases[age] / births[age] * 100000 for age in cases}
indirect = quiet(down.compare_rates, "cases", time="births", for_each_group_in="birth_order", allowing_for="mother_age",
                 standard_rates=rates, per=100000)
indirect.show()
for row in read_rows(indirect):
    checks.check_values(DS, f"{M}, birth order {row['group']}",
                        {quantity: row[quantity] for quantity in ("observed / expected", "expected",
                                                                  "observed / expected 95 % CI low",
                                                                  "observed / expected 95 % CI high", "p-value")})

,group,observed,expected,observed / expected,observed / expected 95 % CI low,observed / expected 95 % CI high,p-value,method
1,1,412.0,217.210807,1.896775,1.718020,2.089075,6.870290e-32,"R epiR::epi.indirectadj(), exact Poisson inter..."
2,2,490.0,260.287755,1.882532,1.719517,2.056836,8.461657e-37,"R epiR::epi.indirectadj(), exact Poisson inter..."
3,3,474.0,265.689129,1.784040,1.627027,1.952114,1.340011e-30,"R epiR::epi.indirectadj(), exact Poisson inter..."
4,4,413.0,213.182511,1.937307,1.754948,2.133467,8.733736e-34,"R epiR::epi.indirectadj(), exact Poisson inter..."
5,5+,740.0,420.629797,1.759267,1.634779,1.890721,6.259492e-45,"R epiR::epi.indirectadj(), exact Poisson inter..."
6,Unknown,225.0,1377.000000,0.163399,0.142744,0.186202,4.940656e-324,"R epiR::epi.indirectadj(), exact Poisson inter..."


agrees  Down syndrome by birth order | compare_rates (standard_rates=), birth order 1 | observed / expected: simplydata 1.89677486668, SciPy, exact Poisson formula 1.89677486668; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy, exact Poisson formula (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  Down syndrome by birth order | compare_rates (standard_rates=), birth order 1 | observed / expected: simplydata 1.89677486668, R epitools + poisson.test 1.89677486668; relative difference 0.00e+00 (allowed 1e-10)
      Source: R epitools + poisson.test (run reference) - https://cran.r-project.org/package=epitools (version 0.5.10.1)
agrees  Down syndrome by birth order | compare_rates (standard_rates=), birth order 1 | expected: simplydata 217.210807269, SciPy, exact Poisson formula 217.210807269; relative difference 1.31e-16 (allowed 1e-10)
      Source: SciPy, exact Poisson formula (run reference) - https://docs.scipy.org/doc/scipy-1.18.

## The 2 x 2 crossover: the bioequivalence reference datasets (Schütz, Labes & Fuglsang 2014)

The EMA model on log AUC with sequence, subject within sequence, period and formulation as fixed effects; the
ratio of geometric means (test / reference) with its 90 % interval and the within-subject CV,
sqrt(exp(residual variance) - 1). Schütz et al. (Table II) print the ratio and the interval in percent with 2
decimals, so the tolerance is half a unit of the last digit (0.00005 as a ratio).

In [13]:
load("bioequivalence_A").compare_averages_of("auc", for_each_group_in="formulation", matched_by="subject",
                                             period="period", same_within=[0.8, 1.25])

"auc": 18 subjects in a 2 x 2 crossover (9 reference then test and 9 test then reference).
Ratio of the geometric means, test / reference: 95.09 % (90 % CI 90.76 % to 99.62 %). Within-subject CV 8.01 %.
The 90 % interval lies inside 80.00 % to 125.00 %, so test and reference are bioequivalent for "auc" (two one-sided tests at 5 % each).
Method: the EMA (2010) model with all effects fixed: statsmodels OLS of log(auc) on sequence, subject (within sequence), period and formulation, as R lm(log(auc) ~ sequence + subject + period + formulation); the 90 % interval taken back from logs; within-subject CV sqrt(exp(MSE) - 1), as R PowerTOST::se2CV().
simplydata does not test for carry-over first: that two-stage procedure biases the result (Freeman 1989), and a washout long enough to rule it out is part of the design.


,term,estimate,90 % CI low,90 % CI high,within-subject CV,subjects,degrees of freedom,margin low,margin high,answer,method
1,ratio of geometric means (test / reference),0.950862,0.907621,0.996162,0.080102,18,16.0,0.8,1.25,same within the margin,"EMA 2 x 2 crossover, all effects fixed, log sc..."


In [14]:
for name in ("A", "B", "C"):
    found = read_row(quiet(load(f"bioequivalence_{name}").compare_averages_of, "auc", for_each_group_in="formulation",
                           matched_by="subject", period="period", same_within=[0.8, 1.25]))
    checks.check_values(f"bioequivalence reference dataset {name}", "compare_averages_of (period=, crossover)",
                        {"ratio": found["estimate"], "90 % CI low": found["90 % CI low"],
                         "90 % CI high": found["90 % CI high"], "within-subject CV": found["within-subject CV"]})

agrees  bioequivalence reference dataset A | compare_averages_of (period=, crossover) | ratio: simplydata 0.950861580106, Schütz et al. (2014), Table II 0.9509; difference 3.84e-05 (allowed 5e-05)
      Source: Schütz et al. (2014), Table II (printed) - https://doi.org/10.1208/s12248-014-9661-0
agrees  bioequivalence reference dataset A | compare_averages_of (period=, crossover) | ratio: simplydata 0.950861580106, statsmodels OLS 0.950861580106; relative difference 3.50e-16 (allowed 1e-09)
      Source: statsmodels OLS (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=OLS
agrees  bioequivalence reference dataset A | compare_averages_of (period=, crossover) | ratio: simplydata 0.950861580106, R lm (EMA model) 0.950861580106; relative difference 9.34e-16 (allowed 1e-09)
      Source: R lm (EMA model) (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  bioequivalence reference dataset A | compare_averages_of (period=, crossover) | 90 % CI 

## Result

In [15]:
results = checks.finish()

16_cause_and_effect: 196 comparisons - 192 agrees, 4 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
